# 쓰레기 검출 YOLO 학습 + Android(LiteRT) 변환 — Google Colab

로컬 PC에 GPU가 없거나 Windows에서 LiteRT 변환이 안 될 때 사용합니다.
런타임 → 런타임 유형 변경 → **GPU(T4)** 선택 후 위에서부터 실행하세요.

1. `ml/` 폴더(datasets 제외)를 zip 으로 올리거나, YOLO 형식 데이터셋 zip 을 올립니다.
2. 학습 → `best.pt`
3. `format=litert nms=None` 으로 변환 → `.tflite` 다운로드 → `app/assets/models/` 에 복사

In [ ]:
!pip -q install "ultralytics>=8.4.142"
import ultralytics; ultralytics.checks()

In [ ]:
# 데이터셋 업로드 (YOLO 형식: images/, labels/, data.yaml 이 든 zip)
from google.colab import files
import zipfile, pathlib
up = files.upload()
zip_name = next(iter(up))
zipfile.ZipFile(zip_name).extractall('dataset')
data_yaml = next(pathlib.Path('dataset').rglob('data.yaml'))
print('data.yaml:', data_yaml)

In [ ]:
# data.yaml 의 path 를 Colab 경로로 고정
import yaml
cfg = yaml.safe_load(open(data_yaml))
cfg['path'] = str(data_yaml.parent.resolve())
yaml.safe_dump(cfg, open(data_yaml, 'w'), allow_unicode=True, sort_keys=False)
print(cfg)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo26n.pt')   # 더 정확하게: yolo26s.pt (느려짐)
results = model.train(data=str(data_yaml), epochs=100, imgsz=640, batch=16, patience=30,
                      project='runs', name='trash', exist_ok=True, seed=0)
best = pathlib.Path(results.save_dir) / 'weights' / 'best.pt'
print('best:', best)

In [ ]:
# Android 용 LiteRT 변환 (플러그인 요구사항: nms=None, imgsz=640)
out = YOLO(str(best)).export(format='litert', nms=None, imgsz=640)
print(out)
tflites = list(pathlib.Path(out).rglob('*.tflite')) if pathlib.Path(out).is_dir() else [pathlib.Path(out)]
print(tflites)

In [ ]:
# 다운로드 → app/assets/models/ 에 복사 후 flutter run
for t in tflites:
    files.download(str(t))
files.download(str(best))